# Feature Engineering — Online Shoppers Purchasing Intention

**Objetivo:** partir del dataset limpio de `ETL.ipynb` (`data/processed/online_shoppers_intention_procesado.csv`) y dejarlo listo para entrenar: decisión sobre las columnas `outlier_*`, features derivadas, split train/test estratificado, y encoding + escalado ajustado solo con train.

Este notebook no entrena ningún modelo: persiste `X_train`, `X_test`, `y_train`, `y_test` y el preprocesador ajustado en `data/models/`, listos para `modeling_mvp.ipynb`.

## 1. Configuración y carga de datos

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

# Agregamos la raíz del proyecto al path para poder importar `src`
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import cargar_datos_procesados, COLUMNAS_CATEGORICAS
from src.features import (
    crear_features_derivadas,
    dividir_train_test,
    construir_column_transformer,
    guardar_artefactos_modelado,
    COLUMNAS_NUMERICAS_MODELADO,
    COLUMNAS_FEATURES_DERIVADAS,
)

df = cargar_datos_procesados()
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

Dataset cargado: 12205 filas x 30 columnas


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,outlier_Administrative,outlier_Administrative_Duration,outlier_Informational,outlier_Informational_Duration,outlier_ProductRelated,outlier_ProductRelated_Duration,outlier_BounceRates,outlier_ExitRates,outlier_PageValues,outlier_SpecialDay,Weekend_label,Revenue_label
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False,False,False,False,False,False,False,True,True,False,False,No,No compra
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False,False,False,False,False,False,False,False,True,False,False,No,No compra
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False,False,False,False,False,False,False,True,True,False,False,No,No compra
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False,False,False,False,False,False,False,True,True,False,False,No,No compra
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False,False,False,False,False,False,False,False,False,False,False,Sí,No compra


Partimos del dataset ya limpio y sin duplicados que produce `ETL.ipynb`; ver ese notebook para el detalle de las decisiones de limpieza (esquema, tipos, duplicados, outliers, consistencia).

## 2. Outliers: qué hacemos con las columnas `outlier_*`

In [2]:
columnas_outlier = [c for c in df.columns if c.startswith("outlier_")]
df[columnas_outlier].sum().sort_values(ascending=False)

outlier_PageValues                 2730
outlier_Informational              2631
outlier_Informational_Duration     2405
outlier_BounceRates                1428
outlier_ExitRates                  1325
outlier_SpecialDay                 1249
outlier_Administrative_Duration    1149
outlier_ProductRelated             1007
outlier_ProductRelated_Duration     951
outlier_Administrative              404
dtype: int64

**Decisión: no se capan.** El EDA y el ETL ya concluyeron que estos valores probablemente reflejan comportamiento real de navegación, no errores de captura. Las columnas `outlier_*` cumplieron su función de auditoría en el ETL; acá se descartan junto con las etiquetas legibles para dashboard (`Weekend_label`, `Revenue_label`, que no aportan como feature) y seguimos trabajando con los valores numéricos originales, sin recortarlos.

In [3]:
df = df.drop(columns=columnas_outlier + ["Weekend_label", "Revenue_label"])
df.shape

(12205, 18)

## 3. Features derivadas

In [4]:
df = crear_features_derivadas(df)
df[COLUMNAS_FEATURES_DERIVADAS].describe()

,duracion_total,paginas_totales,duracion_promedio_pagina,proporcion_paginas_producto
count,12205.000000,12205.000000,12205.000000,12205.000000
mean,1323.454242,34.893240,38.179096,0.904126
std,2043.871589,46.627336,43.472877,0.142281
min,0.000000,0.000000,0.000000,0.000000
25%,231.666667,9.000000,18.555556,0.857143
50%,690.958333,20.000000,29.970787,0.961538
75%,1643.958333,42.000000,45.688278,1.000000
max,69921.647230,746.000000,1411.000000,1.000000


Se agregan cinco features que resumen la sesión más allá de los conteos crudos por tipo de página:

- `duracion_total` y `paginas_totales`: intensidad global de la sesión.
- `duracion_promedio_pagina`: tiempo promedio por página vista (proxy de interés/lectura, no solo de volumen).
- `proporcion_paginas_producto`: qué fracción de la navegación fue a páginas de producto, independiente de cuán larga fue la sesión.
- `es_visitante_recurrente`: booleano derivado de `VisitorType`, más directo para el modelo que la categoría completa.

## 4. Split train/test estratificado

In [5]:
X_train, X_test, y_train, y_test = dividir_train_test(df)
print(f"Train: {X_train.shape}, balance Revenue: {y_train.mean():.4f}")
print(f"Test: {X_test.shape}, balance Revenue: {y_test.mean():.4f}")

Train: (9764, 22), balance Revenue: 0.1563
Test: (2441, 22), balance Revenue: 0.1565


Se estratifica por `Revenue` porque el dataset está desbalanceado (~85/15): sin estratificar, un split al azar podría dejar una proporción de compras distinta en train y en test. El split se hace antes de ajustar cualquier encoder/scaler, para que el conjunto de test no influya en los parámetros aprendidos (evita fuga de datos).

## 5. Encoding y escalado

In [6]:
columnas_numericas = COLUMNAS_NUMERICAS_MODELADO + ["Weekend", "es_visitante_recurrente"]
preprocessor = construir_column_transformer(columnas_numericas, COLUMNAS_CATEGORICAS)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print(f"X_train procesado: {X_train_proc.shape}")
print(f"X_test procesado: {X_test_proc.shape}")

X_train procesado: (9764, 78)
X_test procesado: (2441, 78)


El `ColumnTransformer` se ajusta (`fit`) solo con `X_train` y se reutiliza (`transform`) para `X_test`: las columnas numéricas (originales + derivadas) se escalan con `StandardScaler`, y las categóricas con `OneHotEncoder(handle_unknown="ignore")`, para que categorías nuevas en producción no rompan la inferencia.

## 6. Persistencia de artefactos

In [7]:
ruta_guardada = guardar_artefactos_modelado(
    X_train_proc, X_test_proc, y_train, y_test, preprocessor
)
print(f"Artefactos guardados en: {ruta_guardada}")

Artefactos guardados en: /home/juanma/HENRRY/PROYECTO_FINAL/Proyecto-Final-Henry/data/models


## 7. Conclusiones

A partir del dataset limpio de `ETL.ipynb`, este notebook:

- **Decidió** no capar los outliers marcados por el ETL, y descartó las columnas de auditoría/dashboard que no aportan al modelo.
- **Derivó** cinco features nuevas a partir de las columnas de conteo y duración originales.
- **Separó** train/test de forma estratificada, antes de ajustar cualquier transformación, para evitar fuga de datos.
- **Codificó y escaló** las features (`OneHotEncoder` + `StandardScaler`), ajustando el `ColumnTransformer` solo con train.
- **Persistió** `X_train`, `X_test`, `y_train`, `y_test` y el preprocesador ajustado en `data/models/`.

**Lo que queda para `modeling_mvp.ipynb`:** cargar estos artefactos y entrenar (baseline de regresión logística, manejo del desbalance con SMOTE o `class_weight`, comparación con modelos ensemble, y la lógica del recomendador a partir de la probabilidad estimada).